# 🎬 Live Demo: Portfolio Backtesting System (Complete Setup)

**DADS 4002 - Database Systems Project**

**Notebook นี้รวมทุกอย่างไว้แล้ว - ไม่ต้องรัน setup_procedures.py แยก!**

---

## 🎯 วิธีใช้งาน (สำหรับเครื่องใหม่)

### **เครื่องต้องมี:**
1. ✅ Python 3.8+ ติดตั้งแล้ว
2. ✅ MySQL Server ทำงานอยู่
3. ✅ Database `portfolio_backtesting` สร้างแล้ว (รัน `database/complete_setup.sql`)
4. ✅ Jupyter Notebook ติดตั้งแล้ว: `pip install jupyter mysql-connector-python`

### **ขั้นตอนการใช้งาน:**
1. **แก้ไข MySQL Password** ใน Cell 1 (ด้านล่าง) ให้ตรงกับเครื่องนั้น
2. **รัน Cell 1** (Setup + Install SQL Procedures) → **Shift+Enter**
3. **รัน Cell 2** (Live Demo Menu) → **Shift+Enter ครั้งเดียว**
4. **เลือก Option 1-5** ไปเรื่อยๆ จนกว่าจะเลือก 0 เพื่อออก

---

## ⭐ Features

- ✅ **ไม่ต้องรัน setup_procedures.py แยก** - Notebook นี้ติดตั้ง SQL Procedures อัตโนมัติ
- ✅ **ใช้งานได้บนเครื่องใหม่ทันที** - แค่แก้ password
- ✅ **Interactive Menu** - เลือก Option 1-5 ไปเรื่อยๆ
- ✅ **SQL-based Analytics** - ใช้ Stored Procedures ทั้งหมด

---

---

# 📦 Cell 1: Setup + Install SQL Procedures

## ⚠️ **สำคัญ: แก้ไข MySQL Password ให้ตรงกับเครื่องนี้**

**ก่อนรัน Cell นี้:**
1. หา `MYSQL_CONFIG` ด้านล่าง
2. แก้ไข `'password': 'krittanut123456'` ให้ตรงกับ MySQL password ของเครื่องนี้
3. บันทึก (Ctrl+S)
4. กด **Shift+Enter** เพื่อรัน Cell นี้

---

**Cell นี้จะทำอะไร:**
- เชื่อมต่อ MySQL
- อ่านไฟล์ `database/stored_procedures.sql`
- ติดตั้ง 8 Stored Procedures + 3 Views อัตโนมัติ
- แสดงสถิติของ Database

---

**กด Shift+Enter เพื่อรัน Cell นี้:**

In [ ]:
import mysql.connector
import os
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# ========================================
# ⚠️ แก้ไข PASSWORD ตรงนี้!
# ========================================
MYSQL_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': 'krittanut123456',  # ⬅️ แก้ไขตรงนี้!
    'database': 'portfolio_backtesting'
}
# ========================================

print("="*80)
print("🚀 Portfolio Backtesting System - Complete Setup")
print("="*80)

# Step 1: Connect to MySQL
print("\n🔌 กำลังเชื่อมต่อ MySQL...")
try:
    conn = mysql.connector.connect(**MYSQL_CONFIG)
    cursor = conn.cursor(dictionary=True)
    print("✅ เชื่อมต่อ MySQL สำเร็จ!")
    print(f"📊 Database: {MYSQL_CONFIG['database']}")
except mysql.connector.Error as e:
    print(f"❌ Connection Error: {e}")
    print("\n💡 วิธีแก้:")
    print("   1. ตรวจสอบว่า MySQL Server กำลังทำงานอยู่")
    print("   2. ตรวจสอบ password ให้ถูกต้อง (แก้ไข MYSQL_CONFIG ด้านบน)")
    print("   3. ตรวจสอบว่า database 'portfolio_backtesting' มีอยู่")
    raise

# Step 2: Read SQL file
print("\n📖 กำลังอ่านไฟล์: database/stored_procedures.sql")
sql_file_path = 'database/stored_procedures.sql'

if not os.path.exists(sql_file_path):
    print(f"❌ ไม่พบไฟล์: {sql_file_path}")
    print("💡 กรุณาตรวจสอบว่าอยู่ในโฟลเดอร์ desktop-tutorial")
    raise FileNotFoundError(sql_file_path)

with open(sql_file_path, 'r', encoding='utf-8') as f:
    sql_content = f.read()

print("✅ อ่านไฟล์สำเร็จ")

# Step 3: Parse and execute SQL statements
print("\n⚙️  กำลังติดตั้ง SQL Stored Procedures & Views...")
print("-"*80)

statements = []
current_delimiter = ';'
buffer = ''

for line in sql_content.split('\n'):
    line = line.strip()
    
    # Skip comments and empty lines
    if line.startswith('--') or not line:
        continue
    
    # Check for DELIMITER change
    if line.upper().startswith('DELIMITER'):
        if '//' in line:
            current_delimiter = '//'
        else:
            current_delimiter = ';'
        continue
    
    buffer += line + ' '
    
    # Check if statement ends with current delimiter
    if line.endswith(current_delimiter):
        stmt = buffer.rstrip(current_delimiter).strip()
        if stmt:
            statements.append(stmt)
        buffer = ''

# Execute statements
success_count = 0
error_count = 0

for stmt in statements:
    try:
        # Skip USE database statement
        if stmt.upper().startswith('USE '):
            continue
        
        cursor.execute(stmt)
        success_count += 1
        
        # Show progress
        if 'CREATE VIEW' in stmt.upper():
            view_name = stmt.split('VIEW')[1].split('AS')[0].strip().split()[0]
            print(f"  ✅ สร้าง View: {view_name}")
        elif 'CREATE PROCEDURE' in stmt.upper():
            proc_name = stmt.split('PROCEDURE')[1].split('(')[0].strip()
            print(f"  ✅ สร้าง Procedure: {proc_name}")
        elif 'DROP VIEW' in stmt.upper():
            print(f"  🗑️  Drop existing View (if exists)")
        elif 'DROP PROCEDURE' in stmt.upper():
            print(f"  🗑️  Drop existing Procedure (if exists)")
    
    except mysql.connector.Error as e:
        error_count += 1
        if 'already exists' not in str(e).lower():
            print(f"  ⚠️  Warning: {e}")

conn.commit()

# Step 4: Verify installation
print("\n" + "="*80)
print("📊 สรุปผลการติดตั้ง")
print("="*80)
print(f"✅ สำเร็จ: {success_count} statements")
print(f"❌ ล้มเหลว: {error_count} statements")

# Check Views
cursor.execute("SHOW FULL TABLES WHERE Table_type = 'VIEW'")
views = cursor.fetchall()

print("\n📋 Views ที่ติดตั้งแล้ว:")
if views:
    for view in views:
        view_name = list(view.values())[0]
        print(f"   ✓ {view_name}")
else:
    print("   ⚠️  ไม่พบ Views")

# Check Stored Procedures
cursor.execute("SHOW PROCEDURE STATUS WHERE Db = 'portfolio_backtesting'")
procedures = cursor.fetchall()

print("\n📋 Stored Procedures ที่ติดตั้งแล้ว:")
if procedures:
    for proc in procedures:
        print(f"   ✓ {proc['Name']}")
else:
    print("   ⚠️  ไม่พบ Stored Procedures")

# Step 5: Get database statistics
print("\n" + "="*80)
print("📈 Database Statistics")
print("="*80)

cursor.execute("SELECT COUNT(*) as total FROM etf_master")
etf_count = cursor.fetchone()['total']

cursor.execute("SELECT COUNT(*) as total FROM benchmark_portfolios")
portfolio_count = cursor.fetchone()['total']

cursor.execute("SELECT COUNT(*) as total FROM price_history")
price_count = cursor.fetchone()['total']

print(f"   - ETFs: {etf_count:,}")
print(f"   - Portfolios: {portfolio_count:,}")
print(f"   - Price History Records: {price_count:,}")

print("\n" + "="*80)
print("✅ Setup เสร็จสมบูรณ์!")
print("="*80)
print("\n🎉 พร้อมใช้งาน Live Demo!")
print("💡 กรุณารัน Cell ถัดไป (Main Loop) โดยกด Shift+Enter ครั้งเดียว")
print("="*80)

# Keep connection open for next cell
# DON'T close cursor and conn yet

---

# 🎯 Cell 2: Live Demo - Interactive Analytics Menu

## **กด Shift+Enter เพื่อเริ่ม Demo (ครั้งเดียว)**

**จากนั้น:**
- เลือก Option 1-5 เพื่อ Demo Analytics Features
- เลือก 0 เพื่อออกจากโปรแกรม

---

In [ ]:
def show_menu():
    """แสดง Analytics Menu"""
    print("\n" + "="*80)
    print("📈 Data Analytics Menu (SQL-based)")
    print("="*80)
    print("")
    print("1. 🏆 Top Performers - Top 5 ETFs ที่ดีที่สุด")
    print("2. ⚖️  Portfolio Comparison - เปรียบเทียบ Portfolios ทั้งหมด")
    print("3. 🔗 ETF Correlation - วิเคราะห์ความสัมพันธ์ระหว่าง ETFs")
    print("4. 📊 Sharpe Ratio Calculator - คำนวณ Sharpe Ratio")
    print("5. 📈 Top 10 ETFs Performance - ดูผลตอบแทนจาก SQL View")
    print("0. 🚪 Exit - ออกจากโปรแกรม")
    print("")
    print("="*80)

def top_performers(cursor):
    """Analytics #1: Top Performers using sp_get_top_performers"""
    print("\n🏆 Top 5 ETFs ที่มี Sharpe Ratio สูงสุด")
    print("="*80)
    print("📌 ใช้ SQL Stored Procedure: sp_get_top_performers")
    print("="*80)
    
    try:
        cursor.callproc('sp_get_top_performers', ['sharpe', 5, '2009-01-01', '2025-01-01'])
        
        for result in cursor.stored_results():
            rows = result.fetchall()
        
        if rows:
            print(f"\n{'Rank':<6} {'Ticker':<10} {'ETF Name':<30} {'Return':<12} {'Volatility':<12} {'Sharpe':<10}")
            print("-"*80)
            
            for i, row in enumerate(rows, 1):
                ticker = row['ticker_symbol']
                name = row['etf_name'][:28]
                ret = f"{row['annualized_return_pct']:.2f}%"
                vol = f"{row['annualized_volatility_pct']:.2f}%"
                sharpe = f"{row['sharpe_ratio']:.4f}"
                print(f"#{i:<5} {ticker:<10} {name:<30} {ret:<12} {vol:<12} {sharpe:<10}")
            
            top = rows[0]
            print("\n💡 Actionable Insights:")
            print(f"   - 🎯 {top['ticker_symbol']} มี Sharpe Ratio สูงสุด ({top['sharpe_ratio']:.4f})")
            print(f"   - 📊 ผลตอบแทนต่อปี: {top['annualized_return_pct']:.2f}%")
            print(f"   - 💼 แนะนำสำหรับนักลงทุนที่ต้องการผลตอบแทนดีเมื่อปรับความเสี่ยง")
            if top['sharpe_ratio'] > 1.0:
                print(f"   - ✅ Sharpe Ratio > 1.0 = ผลตอบแทนคุ้มค่ากับความเสี่ยง")
        else:
            print("\n❌ ไม่พบข้อมูล")
    
    except mysql.connector.Error as e:
        print(f"\n❌ Database Error: {e}")

def portfolio_comparison(cursor):
    """Analytics #2: Portfolio Comparison using sp_compare_portfolios"""
    print("\n⚖️  เปรียบเทียบ Benchmark Portfolios")
    print("="*80)
    print("📌 ใช้ SQL Stored Procedure: sp_compare_portfolios")
    print("="*80)
    
    try:
        cursor.callproc('sp_compare_portfolios', ['2020-01-01', '2025-01-01'])
        
        for result in cursor.stored_results():
            rows = result.fetchall()
        
        if rows:
            print(f"\n{'Portfolio':<30} {'Risk Level':<15} {'Return':<12} {'Volatility':<12} {'Sharpe':<10}")
            print("-"*80)
            
            for row in rows:
                portfolio = row['benchmark_name'][:28]
                risk = row['risk_level'][:13]
                ret = f"{row['portfolio_return_pct']:.2f}%"
                vol = f"{row['annualized_volatility_pct']:.2f}%"
                sharpe = f"{row['sharpe_ratio']:.4f}"
                print(f"{portfolio:<30} {risk:<15} {ret:<12} {vol:<12} {sharpe:<10}")
            
            best = max(rows, key=lambda x: x['sharpe_ratio'])
            print("\n💡 Actionable Insights:")
            print(f"   - 🏆 Portfolio ที่ดีที่สุด: {best['benchmark_name']}")
            print(f"   - 📊 Sharpe Ratio: {best['sharpe_ratio']:.4f}")
            print(f"   - 💰 ผลตอบแทนต่อปี: {best['portfolio_return_pct']:.2f}%")
            print(f"   - 🎯 ระดับความเสี่ยง: {best['risk_level']}")
        else:
            print("\n❌ ไม่พบข้อมูล")
    
    except mysql.connector.Error as e:
        print(f"\n❌ Database Error: {e}")

def etf_correlation(cursor):
    """Analytics #3: ETF Correlation using sp_get_etf_correlation"""
    print("\n🔗 ETF Correlation Analysis")
    print("="*80)
    print("📌 ใช้ SQL Stored Procedure: sp_get_etf_correlation")
    print("="*80)
    
    ticker1 = input("\nป้อน Ticker 1 (เช่น SPY): ").strip().upper()
    ticker2 = input("ป้อน Ticker 2 (เช่น QQQ): ").strip().upper()
    
    try:
        result_args = cursor.callproc('sp_get_etf_correlation', [ticker1, ticker2, 0])
        correlation = result_args[2]
        
        if correlation is not None:
            print(f"\n📊 {ticker1} vs {ticker2}")
            print("-"*80)
            print(f"Correlation: {correlation:.4f}")
            
            if correlation > 0.8:
                strength = "แข็งแกร่งมาก (Highly Correlated)"
                recommendation = "⚠️  ไม่แนะนำให้ถือทั้ง 2 ETFs ในพอร์ตเดียวกัน"
                reason = "มีความเสี่ยงคล้ายกันมาก ไม่ได้ช่วย Diversify"
            elif correlation > 0.5:
                strength = "แข็งแกร่งปานกลาง (Moderately Correlated)"
                recommendation = "⚠️  ควรพิจารณาสัดส่วนการถืออย่างรอบคอบ"
                reason = "มีความสัมพันธ์ปานกลาง อาจช่วย Diversify ได้บ้าง"
            elif correlation > 0:
                strength = "อ่อน (Weakly Correlated)"
                recommendation = "✅ เหมาะสำหรับถือร่วมกัน"
                reason = "ช่วย Diversify ความเสี่ยงได้ดี"
            else:
                strength = "ติดลบ (Negative Correlation)"
                recommendation = "✅ ดีมากสำหรับ Diversification"
                reason = "เคลื่อนไหวในทิศทางตรงข้าม ช่วยลดความเสี่ยง"
            
            print(f"\nความสัมพันธ์: {strength}")
            print("\n💡 Actionable Insights:")
            print(f"   - {recommendation}")
            print(f"   - เหตุผล: {reason}")
        else:
            print("\n❌ ไม่พบข้อมูลหรือ Ticker ไม่ถูกต้อง")
            print("💡 ตัวอย่าง Ticker: SPY, QQQ, AGG, GLD, VTI, BND")
    
    except mysql.connector.Error as e:
        print(f"\n❌ Database Error: {e}")

def sharpe_ratio_calculator(cursor):
    """Analytics #4: Sharpe Ratio Calculator using sp_calculate_sharpe_ratio"""
    print("\n📊 Sharpe Ratio Calculator")
    print("="*80)
    print("📌 ใช้ SQL Stored Procedure: sp_calculate_sharpe_ratio")
    print("="*80)
    
    ticker = input("\nป้อน Ticker Symbol (เช่น SPY): ").strip().upper()
    
    try:
        risk_free = float(input("Risk-Free Rate (เช่น 0.02 = 2%): ").strip())
    except:
        risk_free = 0.02
        print(f"ใช้ค่า Default: {risk_free}")
    
    try:
        result_args = cursor.callproc('sp_calculate_sharpe_ratio', [ticker, risk_free, 0])
        sharpe = result_args[2]
        
        if sharpe is not None:
            print(f"\n📊 Sharpe Ratio Analysis: {ticker}")
            print("-"*80)
            print(f"Risk-Free Rate: {risk_free*100:.2f}%")
            print(f"Sharpe Ratio: {sharpe:.4f}")
            
            if sharpe > 2:
                rating = "Excellent (ยอดเยี่ยม)"
            elif sharpe > 1:
                rating = "Good (ดี)"
            elif sharpe > 0.5:
                rating = "Fair (พอใช้)"
            else:
                rating = "Poor (ควรหลีกเลี่ยง)"
            
            print(f"\n💡 Actionable Insights:")
            print(f"   - 🎯 Rating: {rating}")
            if sharpe > 1:
                print(f"   - ✅ Sharpe Ratio > 1.0 = ผลตอบแทนคุ้มค่ากับความเสี่ยง")
                print(f"   - 💼 เหมาะสำหรับการลงทุน")
            else:
                print(f"   - ⚠️  Sharpe Ratio < 1.0 = ควรพิจารณาความเสี่ยงอย่างรอบคอบ")
        else:
            print("\n❌ ไม่พบข้อมูลหรือ Ticker ไม่ถูกต้อง")
            print("💡 ตัวอย่าง Ticker: SPY, QQQ, AGG, GLD, VTI, BND")
    
    except mysql.connector.Error as e:
        print(f"\n❌ Database Error: {e}")

def etf_performance_view(cursor):
    """Analytics #5: ETF Performance using SQL View"""
    print("\n📈 Top 10 ETFs Performance (from SQL View)")
    print("="*80)
    print("📌 ใช้ SQL View: vw_etf_performance")
    print("="*80)
    
    query = """
    SELECT
        ticker_symbol,
        annualized_return_pct,
        annualized_volatility_pct,
        sharpe_ratio_approx
    FROM vw_etf_performance
    ORDER BY sharpe_ratio_approx DESC
    LIMIT 10
    """
    
    try:
        cursor.execute(query)
        rows = cursor.fetchall()
        
        if rows:
            print(f"\n{'Rank':<6} {'Ticker':<10} {'Return':<15} {'Volatility':<15} {'Sharpe Ratio':<15}")
            print("-"*80)
            
            for i, row in enumerate(rows, 1):
                ticker = row['ticker_symbol']
                ret = f"{row['annualized_return_pct']:.2f}%" if row['annualized_return_pct'] else 'N/A'
                vol = f"{row['annualized_volatility_pct']:.2f}%" if row['annualized_volatility_pct'] else 'N/A'
                sharpe = f"{row['sharpe_ratio_approx']:.4f}" if row['sharpe_ratio_approx'] else 'N/A'
                print(f"#{i:<5} {ticker:<10} {ret:<15} {vol:<15} {sharpe:<15}")
            
            print("\n💡 Actionable Insights:")
            print("   - ✅ ข้อมูลนี้คำนวณโดยใช้ SQL Window Functions (LAG, OVER)")
            print("   - ✅ ไม่ใช้ Python/pandas ในการคำนวณ")
            print("   - ✅ ตรงตามโจทย์อาจารย์ข้อ 3: ใช้ SQL เป็นเครื่องมือหลักในการวิเคราะห์")
        else:
            print("\n❌ ไม่พบข้อมูล")
    
    except mysql.connector.Error as e:
        print(f"\n❌ Database Error: {e}")

# Main interactive loop
print("\n" + "="*80)
print("🎬 Live Demo Started!")
print("="*80)
print("💡 กรุณาเลือก Option 1-5 เพื่อ Demo Analytics Features")
print("💡 เลือก 0 เพื่อออกจากโปรแกรม")

# Use connection from previous cell
while True:
    try:
        show_menu()
        choice = input("\nเลือก Option (0-5): ").strip()
        
        if choice == '0':
            print("\n" + "="*80)
            print("🚪 ขอบคุณที่ใช้งาน Portfolio Backtesting System!")
            print("="*80)
            print("\n✅ สรุปการ Demo:")
            print("   - ใช้ SQL Stored Procedures ทั้งหมด (ตามโจทย์ข้อ 3)")
            print("   - มี Actionable Insights ในทุก Feature (ตามโจทย์ข้อ 5f)")
            print("   - ข้อมูลจริงจาก Yahoo Finance (ตามโจทย์ข้อ 4)")
            print("\n💼 ระบบหลักอยู่ที่: main.py (Integrated System)")
            print("\n👋 Goodbye!\n")
            break
        
        elif choice == '1':
            top_performers(cursor)
        
        elif choice == '2':
            portfolio_comparison(cursor)
        
        elif choice == '3':
            etf_correlation(cursor)
        
        elif choice == '4':
            sharpe_ratio_calculator(cursor)
        
        elif choice == '5':
            etf_performance_view(cursor)
        
        else:
            print("\n⚠️  กรุณาเลือก Option 0-5 เท่านั้น")
        
        input("\n⏎ กด Enter เพื่อกลับสู่ Menu...")
        
    except KeyboardInterrupt:
        print("\n\n⚠️  Interrupted by user")
        break
    except Exception as e:
        print(f"\n❌ Error: {e}")
        input("\n⏎ กด Enter เพื่อกลับสู่ Menu...")

# Close connection
cursor.close()
conn.close()
print("\n✅ Database connection closed")

---

# ✅ Demo เสร็จสิ้น

## 📊 สรุป

Notebook นี้แสดงให้เห็นว่า:

### **ข้อ 3: ใช้ SQL เป็นเครื่องมือหลักในการวิเคราะห์** ✅
- ใช้ **5 SQL Stored Procedures**: 
  - `sp_get_top_performers`
  - `sp_compare_portfolios`
  - `sp_get_etf_correlation`
  - `sp_calculate_sharpe_ratio`
  
- ใช้ **SQL View**: `vw_etf_performance`

### **ข้อ 5f: Actionable Insights** ✅
- ทุก Analytics Feature มี **คำแนะนำการลงทุน**
- บอกว่าควรทำอย่างไร ไม่ได้แค่แสดงตัวเลข

### **Interactive Live Demo** ✅
- รัน 2 Cells (Setup + Demo) แล้ววนลูปได้เรื่อยๆ
- เหมาะสำหรับนำเสนอต่ออาจารย์
- **ไม่ต้องรัน setup_procedures.py แยก!**

---

## 🎯 สำหรับการใช้งานบนเครื่องใหม่

1. คัดลอกไฟล์ Notebook นี้ไปยังเครื่องใหม่
2. แก้ไข `MYSQL_CONFIG['password']` ใน Cell 1
3. รัน Cell 1 (Setup) → Shift+Enter
4. รัน Cell 2 (Demo) → Shift+Enter
5. เลือก Option 1-5 ไปเรื่อยๆ

---

**Thank you! 🎉**